# 🔍 Step 3: Enhanced Plagiarism Engine Training v2
## IntegriCheck — Academic Integrity Platform

### 🎯 Is Notebook Ka Kaam Kya Hai?
Yeh notebook **plagiarism detection ke liye ML models train** karta hai:
1. **TF-IDF** — text similarity detection (fast, keyword-based)
2. **MinHash LSH** — near-duplicate detection (copy-paste pakadta hai)
3. **SBERT** — semantic similarity (paraphrasing bhi pakadta hai)

### 📊 v1 vs v2 Upgrades
| Component | v1 | v2 |
|-----------|----|----|
| TF-IDF features | 50k unigrams | **80k unigrams+bigrams** |
| TF weighting | Linear | **Sublinear log(1+tf)** |
| MinHash permutations | 128 | **256** |
| MinHash shingles | Word 3-grams | **Char 5-grams** |
| SBERT encoding | Basic | **Batched + L2 normalized** |

### ⚠️ Prerequisite
**Step 2 pehle run hona chahiye** — `data/processed/master_corpus.csv` exist karna chahiye.

### ⏱️ Runtime (CPU)
- TF-IDF: ~5 min | MinHash: ~10 min | SBERT: ~15-25 min | **Total: ~30 min**

## Cell 1: Libraries Import + Path Setup
**Kya karta hai:** Saari libraries import karta hai aur paths set up karta hai.

### Libraries Kya Karte Hain:
- `sklearn.TfidfVectorizer` — TF-IDF model
- `datasketch.MinHash, MinHashLSH` — Near-duplicate detection
- `sentence_transformers` — SBERT semantic embeddings
- `joblib` — Models save/load karne ke liye (pickle se faster)
- `numpy` — Matrix operations

In [2]:
import os, sys, json, time, re
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
from tqdm import tqdm

# ── Project root set karo (same style as your first cell) ─────────────────────
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

BASE_DIR = PROJECT_ROOT   # consistency ke liye

# ── Paths ─────────────────────────────────────────────────────────────────────
MODELS_DIR    = os.path.join(PROJECT_ROOT, 'data', 'models')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

os.makedirs(MODELS_DIR, exist_ok=True)

# Prerequisite check
corpus_path = os.path.join(PROCESSED_DIR, 'master_corpus.csv')
if not os.path.exists(corpus_path):
    raise FileNotFoundError(
        "❌ master_corpus.csv nahi mila!\n"
        "   Pehle step2_data_collection_v2.ipynb chalaao."
    )

# ── Print setup info ─────────────────────────────────────────────────────────
print("=" * 60)
print("  IntegriCheck — Plagiarism Engine Training v2")
print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

print("\nProject paths:")
print(f"  Root        : {PROJECT_ROOT}")
print(f"  Models      : {MODELS_DIR}")
print(f"  Processed   : {PROCESSED_DIR}")

print(f"\n✅ Corpus found: {corpus_path}")
print("✅ Setup complete!")

  IntegriCheck — Plagiarism Engine Training v2
  Started: 2026-05-11 18:40:57

Project paths:
  Root        : e:\ADD\integricheck_project
  Models      : e:\ADD\integricheck_project\data\models
  Processed   : e:\ADD\integricheck_project\data\processed

✅ Corpus found: e:\ADD\integricheck_project\data\processed\master_corpus.csv
✅ Setup complete!


## Cell 2: Corpus Load + Advanced Text Cleaning
**Kya karta hai:** Master corpus load karta hai aur text clean karta hai model training ke liye.

### Cleaning Steps:
1. **URLs remove** — `https://...` links corpus mein noise create karte hain
2. **Email addresses remove** — personal info bhi noise hai
3. **LaTeX math remove** — `$x^2 + y^2$` jaise formulas similarity match nahi denge
4. **ArXiv IDs remove** — `arXiv:2103.12345` patterns
5. **Special chars remove** — sirf alphanumeric + basic punctuation rakho
6. **Lowercase convert** — case-insensitive matching ke liye

### Quality Filter:
Cleaning ke baad bhi agar 80 words se kam ho → remove karo.
Chhote docs false positives create karte hain.

In [3]:
print("[1/5] 📂 Loading + Cleaning Corpus...")

df = pd.read_csv(corpus_path)
print(f"  Loaded: {len(df):,} documents")
print(f"  Source breakdown: {df['source'].value_counts().to_dict()}")


def advanced_clean(text: str) -> str:
    """
    Plagiarism corpus ke liye advanced text cleaning.
    
    Step by step:
    1. URLs remove
    2. Emails remove  
    3. LaTeX math formulas remove
    4. ArXiv paper IDs remove
    5. Special characters normalize
    6. Whitespace fix karo
    7. Lowercase
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    text = re.sub(r'https?://\S+|www\.\S+', '', text)          # URLs
    text = re.sub(r'\S+@\S+\.\S+', '', text)                   # Emails
    text = re.sub(r'\$\$.*?\$\$', '', text, flags=re.DOTALL)    # LaTeX blocks
    text = re.sub(r'\$.*?\$', '', text)                           # Inline LaTeX
    text = re.sub(r'arXiv:\d+\.\d+', '', text)                  # ArXiv IDs
    text = re.sub(r"[^a-zA-Z0-9\s\'\-\.]", ' ', text)        # Special chars
    text = re.sub(r'[ \t]+', ' ', text)                           # Multiple spaces
    text = re.sub(r'\n+', ' ', text)                              # Newlines
    return text.strip().lower()


print("  Cleaning all texts... (may take 2-3 minutes)")
df['text_clean'] = [
    advanced_clean(t)
    for t in tqdm(df['text'].fillna('').tolist(), desc="  Cleaning", ncols=70)
]

# Quality filter: cleaned text mein bhi 80+ words chahiye
before = len(df)
df = df[df['text_clean'].str.split().str.len() >= 80]
df = df.reset_index(drop=True)
after = len(df)

print(f"\n  Before filter: {before:,} | After filter: {after:,} | Removed: {before-after:,}")
print(f"  ✅ Clean corpus ready: {len(df):,} documents")


[1/5] 📂 Loading + Cleaning Corpus...
  Loaded: 8,069 documents
  Source breakdown: {'student_essay': 5000, 'arxiv': 1944, 'wikipedia': 1125}
  Cleaning all texts... (may take 2-3 minutes)


  Cleaning: 100%|███████████████| 8069/8069 [00:02<00:00, 3379.22it/s]



  Before filter: 8,069 | After filter: 8,069 | Removed: 0
  ✅ Clean corpus ready: 8,069 documents


## Cell 3: Enhanced TF-IDF Vectorizer Training
**Kya karta hai:** TF-IDF model train karta hai — yeh plagiarism detection ka core engine hai.

### TF-IDF Kya Hai?
**Term Frequency-Inverse Document Frequency** — har word ko ek weight deta hai:
- Common words (the, is, and) → low weight
- Rare but meaningful words → high weight

### v2 Improvements:

| Parameter | v1 | v2 | Explanation |
|-----------|----|----|-------------|
| `ngram_range=(1,2)` | (1,1) | **(1,2)** | Bigrams add karo — "machine learning" as one feature |
| `max_features=80000` | 50000 | **80000** | Zyada features = better coverage |
| `sublinear_tf=True` | False | **True** | `log(1+tf)` — high-freq words ki dominance kam karo |
| `min_df=2` | 1 | **2** | Very rare terms (typos) ignore karo |
| `max_df=0.85` | 1.0 | **0.85** | Saare docs mein common terms ignore karo |

### Result:
Jab koi text submit hota hai, isko bhi TF-IDF mein convert karo aur cosine similarity se corpus se compare karo.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("\n[2/5] 🔤 Training Enhanced TF-IDF Vectorizer...")
print("  Parameters:")
print("    ngram_range = (1, 2)    ← Unigrams + Bigrams")
print("    max_features = 80,000   ← Vocabulary size")
print("    sublinear_tf = True     ← log(1+tf) weighting")
print("    min_df = 2              ← Word must appear 2+ times")
print("    max_df = 0.85           ← Skip words in 85%+ docs")

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),          # Unigrams (1,1) + Bigrams (2,2)
    max_features=80_000,         # Total vocabulary size
    min_df=2,                    # Min document frequency
    max_df=0.85,                 # Max document frequency (85%)
    sublinear_tf=True,           # log(1+tf) — better for long docs
    strip_accents='unicode',     # é → e, etc.
    analyzer='word',
    token_pattern=r'\b[a-zA-Z][a-zA-Z0-9]{2,}\b',  # Min 3-char tokens
    smooth_idf=True,             # IDF mein zero division avoid
)

print("\n  Fitting TF-IDF on corpus... (5-8 minutes)")
texts        = df['text_clean'].tolist()
tfidf_matrix = tfidf_vectorizer.fit_transform(
    tqdm(texts, desc="  TF-IDF fit", ncols=70)
)

sparsity = 1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])

print(f"\n  TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"  Non-zero elements: {tfidf_matrix.nnz:,}")
print(f"  Sparsity: {sparsity:.3f} (high sparsity = normal for text)")

# Save both vectorizer and matrix
joblib.dump(tfidf_vectorizer,
            os.path.join(MODELS_DIR, 'tfidf_vectorizer.pkl'), compress=3)
joblib.dump(tfidf_matrix,
            os.path.join(MODELS_DIR, 'tfidf_matrix.pkl'),     compress=3)

print(f"\n  ✅ TF-IDF Vectorizer saved")
print(f"  ✅ TF-IDF Matrix saved")
print(f"  Vocabulary size: {len(tfidf_vectorizer.vocabulary_):,} features")

# Quick test — kuch sample words dekhte hain
sample_bigrams = [(k,v) for k,v in tfidf_vectorizer.vocabulary_.items() if ' ' in k][:5]
print(f"\n  Sample bigrams in vocabulary: {[k for k,v in sample_bigrams]}")



[2/5] 🔤 Training Enhanced TF-IDF Vectorizer...
  Parameters:
    ngram_range = (1, 2)    ← Unigrams + Bigrams
    max_features = 80,000   ← Vocabulary size
    sublinear_tf = True     ← log(1+tf) weighting
    min_df = 2              ← Word must appear 2+ times
    max_df = 0.85           ← Skip words in 85%+ docs

  Fitting TF-IDF on corpus... (5-8 minutes)


  TF-IDF fit: 100%|█████████████| 8069/8069 [00:04<00:00, 1868.28it/s]



  TF-IDF Matrix shape: (8069, 80000)
  Non-zero elements: 2,431,959
  Sparsity: 0.996 (high sparsity = normal for text)

  ✅ TF-IDF Vectorizer saved
  ✅ TF-IDF Matrix saved
  Vocabulary size: 80,000 features

  Sample bigrams in vocabulary: ['state space', 'space model', 'model for', 'for event', 'guided image']


## Cell 4: Enhanced MinHash LSH Index
**Kya karta hai:** MinHash LSH (Locality Sensitive Hashing) index banata hai — yeh **copy-paste aur near-duplicate text** pakadta hai.

### MinHash Kya Hai?
- Document ko ek compact "fingerprint" mein convert karta hai
- Do documents ke similar fingerprints → high similarity
- **Exactly same text nahi chahiye** — 70-80% similar bhi pakad leta hai

### v2 Upgrade — Char 5-grams vs Word 3-grams:
```
Word 3-gram: ["machine learning is", "learning is a", "is a powerful"]
Char 5-gram: ["machi", "achine", "chine", "ine l", "ne le", "e lea"]
```
**Char 5-grams better kyun?**
- Word order change karke paraphrase karo → word 3-grams fail
- Char 5-grams thoda bhi original text rakhe toh pakad leta hai
- Spelling mistakes bhi partially handle karta hai

### Parameters:
- `n_perm=256` — 256 random hash functions (v1 mein 128 tha) → better accuracy
- `threshold=0.35` — 35% Jaccard similarity pe match (v1 mein 50% tha) → more sensitive

In [5]:
from datasketch import MinHash, MinHashLSH

print("\n[3/5] 🔗 Building Enhanced MinHash LSH Index...")

# Config
N_PERM       = 256     # Hash permutations (v1: 128, v2: 256 = better)
LSH_THRESH   = 0.35    # Jaccard threshold (v1: 0.5, v2: 0.35 = more sensitive)
SHINGLE_SIZE = 5       # Character n-gram size (v1: word 3-gram, v2: char 5-gram)

print(f"  Config:")
print(f"    n_perm (permutations)  = {N_PERM}   (v1: 128)")
print(f"    threshold (Jaccard)    = {LSH_THRESH}  (v1: 0.50)")
print(f"    shingle_size (char)    = {SHINGLE_SIZE}   (v1: word 3-gram)")


def text_to_minhash(text: str, n_perm: int = N_PERM,
                    shingle_size: int = SHINGLE_SIZE) -> MinHash:
    """
    Text ko MinHash fingerprint mein convert karo.
    
    Process:
    1. Text lowercase + normalize
    2. Character n-grams (shingles) extract karo
    3. Har shingle ko hash karo
    4. MinHash signature banao
    """
    m = MinHash(num_perm=n_perm)
    
    # Normalize
    text_clean = re.sub(r'\s+', ' ', text.lower()).strip()
    
    if len(text_clean) < shingle_size:
        # Too short — full text hash karo
        m.update(text_clean.encode('utf8'))
        return m
    
    # Character n-grams extract karo
    # Example: "hello" with size=3 → ["hel", "ell", "llo"]
    for i in range(len(text_clean) - shingle_size + 1):
        shingle = text_clean[i:i + shingle_size]
        m.update(shingle.encode('utf8'))   # Har shingle hash karo
    
    return m


# LSH Index create karo
lsh      = MinHashLSH(threshold=LSH_THRESH, num_perm=N_PERM)
minhashes = {}

print(f"\n  Building MinHash signatures for {len(df):,} docs...")
print(f"  (Estimated time: 8-12 minutes)")

failed = 0
with tqdm(total=len(df), desc="  MinHash", ncols=70) as pbar:
    for idx, row in df.iterrows():
        doc_id = str(row.get('doc_id', f'doc_{idx}'))
        text   = str(row.get('text', ''))
        
        if len(text) < 20:
            pbar.update(1)
            continue
        
        try:
            mh = text_to_minhash(text)
            minhashes[doc_id] = mh
            lsh.insert(doc_id, mh)        # LSH index mein add karo
        except Exception:
            failed += 1
        
        pbar.update(1)

print(f"\n  Documents indexed: {len(minhashes):,}")
print(f"  Failed (skipped):  {failed:,}")

# Save
joblib.dump(minhashes, os.path.join(MODELS_DIR, 'minhashes.pkl'), compress=3)
joblib.dump(lsh,       os.path.join(MODELS_DIR, 'lsh_index.pkl'), compress=3)

print(f"  ✅ MinHash signatures saved")
print(f"  ✅ LSH Index saved")



[3/5] 🔗 Building Enhanced MinHash LSH Index...
  Config:
    n_perm (permutations)  = 256   (v1: 128)
    threshold (Jaccard)    = 0.35  (v1: 0.50)
    shingle_size (char)    = 5   (v1: word 3-gram)

  Building MinHash signatures for 8,069 docs...
  (Estimated time: 8-12 minutes)


  MinHash: 100%|██████████████████| 8069/8069 [02:55<00:00, 45.90it/s]



  Documents indexed: 8,069
  Failed (skipped):  0
  ✅ MinHash signatures saved
  ✅ LSH Index saved


## Cell 5: SBERT Semantic Embeddings (CPU Optimized)
**Kya karta hai:** SBERT (Sentence-BERT) se har document ka semantic embedding banata hai.

### SBERT Kya Hai?
TF-IDF sirf exact/similar words dhundta hai. SBERT **meaning** samajhta hai:
- "The dog chased the cat" 
- "A canine pursued a feline"  
→ TF-IDF: No match | SBERT: HIGH match ✅

### CPU Optimization:
| Setting | Value | Reason |
|---------|-------|--------|
| `batch_size=32` | 32 docs | OOM avoid karo |
| `SBERT_MAXLEN=256` | 256 chars | Speed — full text slow hoga |
| `normalize_embeddings=True` | True | L2 normalize → cosine = dot product (faster) |
| `device='cpu'` | cpu | GPU nahi hai |

### Model: `all-MiniLM-L6-v2`
- 22M parameters — lightweight
- 384-dimensional embeddings
- Good speed/accuracy balance for CPU

In [6]:
print("\n[4/5] 🧠 Building SBERT Corpus Embeddings (CPU mode)...")

SBERT_BATCH  = 32    # Batch size (CPU ke liye safe)
SBERT_MAXLEN = 256   # Max chars per document (speed ke liye)
MODEL_NAME   = 'all-MiniLM-L6-v2'  # Lightweight + accurate

print(f"  Model: {MODEL_NAME}")
print(f"  Batch size: {SBERT_BATCH} | Max chars: {SBERT_MAXLEN}")

SBERT_OK = False

try:
    from sentence_transformers import SentenceTransformer
    
    print(f"  Loading model... (first time mein download hoga ~90MB)")
    model = SentenceTransformer(MODEL_NAME, device='cpu')
    
    # Har doc ke liye: title + first 256 chars of text
    # Yeh approach full text se fast hai aur accuracy similar hai
    sbert_texts = []
    for _, row in df.iterrows():
        title = str(row.get('title', ''))[:80]
        text  = str(row.get('text_clean', ''))[:SBERT_MAXLEN]
        sbert_texts.append((title + ' ' + text).strip())
    
    n_docs = len(sbert_texts)
    est_minutes = n_docs // SBERT_BATCH * 2 // 60
    print(f"\n  Encoding {n_docs:,} texts...")
    print(f"  ⏱ Estimated time: ~{est_minutes} min on CPU (be patient!)")
    
    corpus_embeddings = model.encode(
        sbert_texts,
        batch_size=SBERT_BATCH,
        show_progress_bar=True,     # Progress bar dikhao
        convert_to_numpy=True,      # NumPy array chahiye
        normalize_embeddings=True,  # L2 normalize → cosine sim = dot product
    )
    
    # Save
    np.save(os.path.join(MODELS_DIR, 'corpus_embeddings.npy'), corpus_embeddings)
    
    print(f"\n  Embeddings shape: {corpus_embeddings.shape}")
    print(f"  Embedding dimension: {corpus_embeddings.shape[1]} (384 for MiniLM)")
    print(f"  ✅ SBERT embeddings saved!")
    SBERT_OK = True

except Exception as e:
    print(f"  ⚠ SBERT failed: {e}")
    print(f"  → Saving placeholder (TF-IDF + MinHash will still work)")
    # Zero placeholder save karo — engine crash na ho
    placeholder = np.zeros((len(df), 384), dtype=np.float32)
    np.save(os.path.join(MODELS_DIR, 'corpus_embeddings.npy'), placeholder)
    print(f"  → Placeholder saved. Rerun this cell later to fix SBERT.")

print(f"\n  SBERT status: {'✅ Working' if SBERT_OK else '⚠ Using placeholder'}")



[4/5] 🧠 Building SBERT Corpus Embeddings (CPU mode)...
  Model: all-MiniLM-L6-v2
  Batch size: 32 | Max chars: 256
  Loading model... (first time mein download hoga ~90MB)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


  Encoding 8,069 texts...
  ⏱ Estimated time: ~8 min on CPU (be patient!)


Batches:   0%|          | 0/253 [00:00<?, ?it/s]


  Embeddings shape: (8069, 384)
  Embedding dimension: 384 (384 for MiniLM)
  ✅ SBERT embeddings saved!

  SBERT status: ✅ Working


## Cell 6: Validation Test
**Kya karta hai:** Trained plagiarism engine ko test karta hai — check karta hai ki sahi kaam kar raha hai ya nahi.

### 3 Test Cases:
1. **Known academic text** → Should be HIGH match (corpus mein hai)
2. **Original creative text** → Should be LOW/NO match
3. **Paraphrased academic text** → Should be MEDIUM match (SBERT pakadega)

### Pass Criteria:
- Test 1 score > Test 2 score ✅
- Test 1 score > 10% (kuch match toh milna chahiye)
- Test 2 score < 20% (original text match nahi hona chahiye)

In [7]:
print("\n[5/5] ✅ Validating Plagiarism Engine...")

# Path adjust karo taaki engine import ho sake
sys.path.insert(0, BASE_DIR)

# Force fresh reload
import importlib
for mod_name in list(sys.modules.keys()):
    if 'plagiarism' in mod_name or 'engine' in mod_name:
        del sys.modules[mod_name]

from src.plagiarism.engine import analyze_plagiarism

# Test 1: Academic text (known content)
test_academic = (
    "Machine learning is a subset of artificial intelligence that enables systems "
    "to learn and improve from experience without being explicitly programmed. "
    "Deep learning models especially convolutional neural networks are highly "
    "effective for image classification tasks using large datasets."
)

# Test 2: Original unrelated text (should be low)
test_original = (
    "My grandmother makes the best chai on cold winter mornings. She adds "
    "cardamom and ginger, and the whole house smells wonderful. We sit on "
    "the veranda watching the birds in the garden while drinking it slowly."
)

# Test 3: Paraphrased academic text
test_paraphrased = (
    "ML systems can automatically improve through training data without direct "
    "programming. Neural networks designed for images have proven extremely "
    "powerful in recognizing objects within photographs and video frames."
)

print("\n  Running test analyses...")
r1 = analyze_plagiarism(test_academic)
r2 = analyze_plagiarism(test_original)
r3 = analyze_plagiarism(test_paraphrased)

print(f"\n  Test 1 — Known academic text:")
print(f"    Score: {r1['final_score']}% | Mode: {r1.get('_mode','?')} | Risk: {r1['risk_level']}")

print(f"\n  Test 2 — Original creative text:")
print(f"    Score: {r2['final_score']}% | Mode: {r2.get('_mode','?')} | Risk: {r2['risk_level']}")

print(f"\n  Test 3 — Paraphrased academic text:")
print(f"    Score: {r3['final_score']}% | Mode: {r3.get('_mode','?')} | Risk: {r3['risk_level']}")

# Validation checks
print("\n  Validation checks:")
check1 = r1['final_score'] > r2['final_score']
check2 = r2['final_score'] < 20
check3 = r1['final_score'] > 5

print(f"    Academic > Original:  {'✅ PASS' if check1 else '❌ FAIL'}")
print(f"    Original < 20%:       {'✅ PASS' if check2 else '❌ FAIL'}")
print(f"    Academic detected:    {'✅ PASS' if check3 else '❌ FAIL'}")

if all([check1, check2, check3]):
    print("\n  🎉 ALL CHECKS PASSED — Engine working correctly!")
else:
    print("\n  ⚠ Some checks failed — check corpus quality")



[5/5] ✅ Validating Plagiarism Engine...

  Running test analyses...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


  Test 1 — Known academic text:
    Score: 43.53% | Mode: ? | Risk: MEDIUM

  Test 2 — Original creative text:
    Score: 15.69% | Mode: ? | Risk: LOW

  Test 3 — Paraphrased academic text:
    Score: 34.6% | Mode: ? | Risk: LOW

  Validation checks:
    Academic > Original:  ✅ PASS
    Original < 20%:       ✅ PASS
    Academic detected:    ✅ PASS

  🎉 ALL CHECKS PASSED — Engine working correctly!


## ✅ Step 3 Complete!

### Models Saved:
| File | Size (approx) | Purpose |
|------|---------------|---------|
| `data/models/tfidf_vectorizer.pkl` | ~50 MB | TF-IDF vocabulary |
| `data/models/tfidf_matrix.pkl` | ~200 MB | Corpus TF-IDF matrix |
| `data/models/minhashes.pkl` | ~100 MB | MinHash signatures |
| `data/models/lsh_index.pkl` | ~30 MB | LSH lookup index |
| `data/models/corpus_embeddings.npy` | ~100 MB | SBERT embeddings |

### How They Work Together:
```
Query Text
    │
    ├─► TF-IDF similarity  ─► Top 10 keyword matches
    ├─► MinHash LSH         ─► Near-duplicate matches  
    └─► SBERT similarity   ─► Semantic/paraphrase matches
                │
                ▼
          Combined Score (weighted average)
```

### ▶️ Next Step:
`step4_ai_detection_v2.ipynb` chalaao — AI detection ensemble train hoga!

In [8]:
print("\n" + "="*60)
print("  STEP 3 COMPLETE ✅")
print("="*60)

# Model sizes check
model_files = [
    'tfidf_vectorizer.pkl', 'tfidf_matrix.pkl',
    'minhashes.pkl', 'lsh_index.pkl', 'corpus_embeddings.npy'
]
print("\n  Saved models:")
total_size = 0
for f in model_files:
    path = os.path.join(MODELS_DIR, f)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        total_size += size
        print(f"    ✅ {f:35s} {size:.1f} MB")
    else:
        print(f"    ❌ {f} NOT FOUND")

print(f"\n  Total model size: {total_size:.1f} MB")
print(f"\n  ▶ Next: Run step4_ai_detection_v2.ipynb")
print("="*60)



  STEP 3 COMPLETE ✅

  Saved models:
    ✅ tfidf_vectorizer.pkl                0.9 MB
    ✅ tfidf_matrix.pkl                    19.8 MB
    ✅ minhashes.pkl                       7.8 MB
    ✅ lsh_index.pkl                       6.5 MB
    ✅ corpus_embeddings.npy               11.8 MB

  Total model size: 46.8 MB

  ▶ Next: Run step4_ai_detection_v2.ipynb
